# Generics for Reusable ADTs

You have been using generics since Chapter 8: `List<T>` and `Dictionary<TKey, TValue>`. In an ADT, generics let one contract and one implementation work with many element types while preserving compile-time type checking.

For the caller obligations of the ref swap exercise, review {ref}`Chapter 5 parameter passing <gap-return-values-out-and-ref>`.


```{contents}
:local:
:depth: 2
```

```{index} generic type
```



```{index} Why Generics Exist
```

## Why Generics Exist



Without generics, a stack that holds `int` would need to be a different class from a stack that holds `string`, or it would have to store everything as `object` and lose type safety. Generics let one implementation work for many types:

```csharp
// Non-generic — stores object, loses type safety
var list = new ArrayList();
list.Add(1);
list.Add("oops");  // compiles — bug at runtime

// Generic — type-safe at compile time
var nums = new List<int>();
nums.Add(1);
// nums.Add("oops");  // compile error
```

For related explanation, see the further reading below.[^development-reading]

The difference between the two blocks is *when* the mistake is discovered. With `ArrayList`, the string goes in without complaint and the failure arrives later, in whatever code reads the element and casts it back to `int`, possibly far from the line that caused it. With `List<int>`, the compiler rejects the insertion at the line where it is written. The generic version also removes the cast: `nums[0]` is already an `int`, so arithmetic on it needs no conversion and no boxing of the value into an object. The example below makes both halves concrete: an object-based cell accepts a wrong value and fails only when it is read, while the generic cell will not compile with a wrong value at all.

In [ ]:
using System;

// Object-based storage: anything goes in, and the error surfaces on the way out.
var loose = new ObjectCell();
loose.Set(1250);
loose.Set("twelve fifty");            // compiles: a string is an object
try
{
    int balance = (int)loose.Get();   // the cast fails here, not at the Set
    Console.WriteLine(balance);
}
catch (InvalidCastException ex)
{
    Console.WriteLine("Runtime failure: " + ex.Message);
}

// Generic storage: the wrong value is rejected before the program runs.
var typed = new Cell<int>(1250);
// typed.Set("twelve fifty");        // compile error CS1503: cannot convert string to int
typed.Set(1300);
Console.WriteLine("Typed balance: " + (typed.Get() + 50));   // no cast needed

class ObjectCell
{
    private object value = 0;
    public object Get() => value;
    public void Set(object newValue) => value = newValue;
}

class Cell<T>
{
    private T value;
    public Cell(T initialValue) => value = initialValue;
    public T Get() => value;
    public void Set(T newValue) => value = newValue;
}

Runtime failure: Unable to cast object of type 'System.String' to type 'System.Int32'.
Typed balance: 1350


The first line reports a runtime failure: the message says a `System.String` cannot be cast to `System.Int32`. Nothing stopped `Set("twelve fifty")`; the damage became visible only at the read, which in a larger program might be in a different file and run minutes later. The second line prints `Typed balance: 1350` with no cast in sight. Uncomment the marked line and the program does not compile, which is the point: a generic container moves a whole class of mistakes from testing time to compile time. The cost is one type argument at the point of construction, and the benefit is that every read is already the right type.

```{index} Reuse an Algorithm Without Erasing Its Types
```



```{index} Reuse an Algorithm Without Erasing Its Types
```

## Reuse an Algorithm Without Erasing Its Types



An undo history of product codes and a history of numeric adjustments can follow the same push/pop rules. Writing two unrelated implementations repeats the same invariant and creates two places to repair a defect. A generic type makes the element type a parameter while leaving the behavioral algorithm shared.

Read `History<T>` as a type definition with an element-type placeholder. Read `History<string>` as one constructed type whose element operations must accept strings. The placeholder is not a variable containing an element and is not a request to decide the type afresh on every assignment. A History<int> and a History<string> are different constructed types even though their operation names are the same.

The general class pattern is `class Container<T> { /* members that use T */ }`. A generic method has its own type parameter list, as in `static T Identity<T>(T value)`. These are declaration patterns, not complete programs. In concrete use the compiler checks compatible argument/result types. That lets a bad element insertion be rejected before a client discovers an incompatible value during a later cast.




```{index} State Which Operations the Generic Code Needs
```



```{index} State Which Operations the Generic Code Needs
```

## State Which Operations the Generic Code Needs



An unconstrained T supports being stored, passed and returned. It does not promise a Price property or a meaningful greater-than operation. If the algorithm needs ordering, supply an ordering policy or an appropriate constraint. If it needs key identity, supply an equality/hash policy with its consistency requirements. Ordering and equality are related concepts but different contracts.

| Algorithm purpose | Information needed beyond storage |
| --- | --- |
| Return the most recently stored item | No ordering comparison |
| Sort values for a report | A consistent ordering policy |
| Find a key in a hash table | Equality and compatible hashing |
| Call an element-specific capability | An explicit capability constraint |

A comparer can say that two invoices have equal totals even when they are distinct invoices. A key-equality policy might instead use invoice ID. Choosing a generic collection does not choose those business meanings automatically. Avoid solving a missing requirement by converting every value to text and comparing that text; it can change numeric order and lose distinctions.

Guided design: create a generic history interface for values that never need to be compared. Would requiring all element types to implement an ordering interface help? It would unnecessarily reject useful element types. Add constraints because an operation needs them, not because generics seem advanced. The short lambda/query forms later in this lesson are a bridge for reading policy arguments; the fundamental generic contract does not depend on mastering LINQ.


```{index} Generic Methods
```



```{index} Generic Methods
```

## Generic Methods



A **generic method** declares one or more *type parameters* in angle brackets after its name and then uses them like ordinary types in its signature and body:

```csharp
static TResult Name<T>(T parameter, ...)   // T stands for a type chosen at each call
```

The caller usually does not write the type argument: the compiler infers `T` from the arguments. `Repeat` below builds an array of `count` copies of `value`, and the same body serves `int` and `string`:


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
// A generic method works for any type T
static T[] Repeat<T>(T value, int count)
{
    var arr = new T[count];
    for (int i = 0; i < count; i++) arr[i] = value;
    return arr;
}

int[]    ints    = Repeat(0, 5);       // [0, 0, 0, 0, 0]
string[] strings = Repeat("hi", 3);   // ["hi", "hi", "hi"]

Console.WriteLine(string.Join(", ", ints));     // 0, 0, 0, 0, 0
Console.WriteLine(string.Join(", ", strings));  // hi, hi, hi

0, 0, 0, 0, 0
hi, hi, hi


The output is `0, 0, 0, 0, 0` and `hi, hi, hi`. For the first call the compiler infers `T` as `int` from the argument `0`, so the method returns `int[]`; for the second it infers `string`. You can name the type argument explicitly, `Repeat<double>(1.5, 2)`, which is required when inference has nothing to go on. The mistake to watch for is mixing types in one call: `Repeat(0, "3")` does not compile because `count` is declared `int`, not `T`.


```{index} Code Completion: A Generic Last Element
```

### Code Completion: A Generic Last Element

Complete `LastOr<T>` so that it returns the last element of an array, or the supplied fallback when the array is empty. The method must work for any element type without a constraint, so it may only store, index, and return values of type `T`. The expected output is `D`, then `none`, then `34`.

In [ ]:
using System;

Console.WriteLine(LastOr(new[] { "A", "B", "C", "D" }, "none"));
Console.WriteLine(LastOr(new string[0], "none"));
Console.WriteLine(LastOr(new[] { 5, 8, 13, 21, 34 }, -1));

static T LastOr<T>(T[] items, T fallback)
{
    // Your code starts here.

    // Your code ends here.
}

In [ ]:
// Solution
using System;

Console.WriteLine(LastOr(new[] { "A", "B", "C", "D" }, "none"));
Console.WriteLine(LastOr(new string[0], "none"));
Console.WriteLine(LastOr(new[] { 5, 8, 13, 21, 34 }, -1));

static T LastOr<T>(T[] items, T fallback)
{
    if (items.Length == 0) return fallback;
    return items[items.Length - 1];
}

D
none
34


Both the array element and the fallback have type `T`, so the compiler can infer `T` from either argument and both return statements type-check. A tempting wrong completion is `return items[^1] ?? fallback;`, which does not compile for an unconstrained `T` because the null-coalescing operator needs to know that `T` can be null; the length check is the version that works for every type.

```{index} generic class
```



```{index} Generic Classes
```

## Generic Classes



A **generic class** puts the type parameter on the class instead of a method, so every member can use it. The type argument is chosen once, when an object is created, and fixes the type of that object's contents for its whole life:

```csharp
class Cell<T>
{
    private T value;                      // a field of the chosen type
    public Cell(T initialValue) { ... }   // the constructor takes that type
    public T Get() { ... }                // members return and accept it
}

Cell<int> score = new Cell<int>(95);      // T is int for this object
```


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
var score = new Cell<int>(95);
var name = new Cell<string>("Ada");

score.Set(98);
Console.WriteLine(score.Get());
Console.WriteLine(name.Get());

// A minimal generic cell ADT
class Cell<T>
{
    private T value;

    public Cell(T initialValue) => value = initialValue;
    public T Get() => value;
    public void Set(T newValue) => value = newValue;
}

98
Ada


The output is `98` and `Ada`. `score` can only ever hold an `int`: `score.Set("x")` is a compile error, which is the type safety that an `object`-based cell cannot offer. The common mistake is writing `new Cell(95)` without the type argument; unlike a method call, a class has no arguments to infer from, so the compiler asks for `Cell<int>` explicitly. The library's `List<T>`, `Dictionary<TKey, TValue>`, and `Queue<T>` are generic classes of exactly this kind.


```{index} type constraint
```



```{index} Type Constraints
```

## Type Constraints




Use `where T : ...` to restrict what types `T` can be. Constraints are useful when an ADT operation needs a capability that not every type has.

| Constraint | Meaning |
|---|---|
| `where T : class` | T must be a reference type |
| `where T : struct` | T must be a value type |
| `where T : new()` | T must have a parameterless constructor |
| `where T : ISomeInterface` | T must implement the interface |
| `where T : BaseClass` | T must inherit from `BaseClass` |

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
// Constraint: T must implement IComparable<T> so we can compare
static T Max<T>(T a, T b) where T : IComparable<T>
    => a.CompareTo(b) >= 0 ? a : b;

Console.WriteLine(Max(3, 7));          // 7
Console.WriteLine(Max("apple", "banana")); // banana

7
banana


The output is `7` and `banana`. Without the `where T : IComparable<T>` clause the body does not compile: the compiler cannot assume an arbitrary `T` has `CompareTo`. The constraint is a promise the caller must keep, so `Max(3, 7)` and `Max("apple", "banana")` are accepted while a call with a type that cannot compare itself is rejected at compile time. Strings compare in ordinal order here, so `"Zebra"` would rank before `"apple"`; Chapter 16 returns to ordering policies.


```{index} Which Constraint Does the Body Need?
```

### Which Constraint Does the Body Need?

For each generic method body below, decide which `where` clause, if any, is required for it to compile, then open the answer.

```csharp
static bool SameAs<T>(T a, T b) => a.Equals(b);                    // 1
static T Larger<T>(T a, T b) => a.CompareTo(b) > 0 ? a : b;        // 2
static T Fresh<T>() => new T();                                     // 3
static int Count<T>(T[] items) => items.Length;                     // 4
static string Label<T>(T item) => item.Describe();                  // 5
```

```{dropdown} Answer
1. No constraint. Every type inherits `Equals(object)` from `System.Object`, so the call compiles for any `T` (though a `where T : IEquatable<T>` constraint would give a typed, non-boxing comparison).
2. `where T : IComparable<T>`. An unconstrained `T` has no `CompareTo`.
3. `where T : new()`. The compiler needs a promise that `T` has a public parameterless constructor.
4. No constraint. The body touches only the array, never an element.
5. `where T : IDescribable` for some interface declaring `string Describe()`. A method name that is not on `object` must come from a constraint.

The rule behind all five: inside the method, `T` offers exactly the members of `object` plus whatever the constraints promise, and nothing else.
```

```{index} generic interface
```



```{index} Generic Interfaces
```

## Generic Interfaces




An ADT contract can be generic too. `IStack<T>` describes stack behavior for any element type `T`, while each implementation decides how to store those elements.

```csharp
interface IStack<T>
{
    int Count { get; }
    void Push(T item);
    T Pop();
    T Peek();
}
```

This is the same idea behind library interfaces such as `IEnumerable<T>`, `ICollection<T>`, and `IList<T>`: the interface names the behavior, and the type parameter names the element type.

The short implementation below keeps its elements in an array that doubles when full, the dynamic-array idea from Chapter 14, and shows that one class can satisfy `IStack<T>` for every `T` at once:


In [ ]:
using System;

IStack<string> tasks = new ArrayStack<string>();
tasks.Push("draft"); tasks.Push("review"); tasks.Push("ship");
Console.WriteLine($"{tasks.Count} tasks, next is {tasks.Peek()}");
while (tasks.Count > 0) Console.WriteLine($"popped {tasks.Pop()}");

interface IStack<T>
{
    int Count { get; }
    void Push(T item);
    T Pop();
    T Peek();
}

class ArrayStack<T> : IStack<T>
{
    private T[] items = new T[2];
    public int Count { get; private set; }

    public void Push(T item)
    {
        if (Count == items.Length) Array.Resize(ref items, items.Length * 2);
        items[Count] = item;
        Count++;
    }

    public T Pop()
    {
        if (Count == 0) throw new InvalidOperationException("The stack is empty.");
        Count--;
        return items[Count];
    }

    public T Peek()
    {
        if (Count == 0) throw new InvalidOperationException("The stack is empty.");
        return items[Count - 1];
    }
}


3 tasks, next is ship
popped ship
popped review
popped draft


The output is `3 tasks, next is ship`, then `popped ship`, `popped review`, `popped draft`. The variable is declared as `IStack<string>`, so the code that uses it depends only on the contract; a linked implementation of the same interface could replace `ArrayStack` without changing those lines. Chapter 15 develops both implementations and their costs.


```{index} Records, Keys, and Homogeneity
```

### Records, Keys, and Homogeneity

A generic collection is *homogeneous*: every element has the one type named by the type argument. That is a feature, not a limitation. A `List<Invoice>` can never hand back something that is not an invoice, so loops over it need no checks. When a collection must mix kinds, the usual answer is still one type, an interface or base class that all the kinds share, as Chapter 12 showed with `IShape`, rather than falling back to `object`.

Many business collections store *records* that are found by a *key*: a customer located by ID, a product by SKU, an order by number. The record carries the key as one of its fields, and the collection needs two pieces of information per entry: the key it searches by and the value it returns. The library expresses this with a second type parameter, `Dictionary<TKey, TValue>` and the `KeyValuePair<TKey, TValue>` it hands out during enumeration. The same idea in a hand-written generic class takes two type parameters:

```csharp
class Entry<TKey, TValue>
{
    public TKey Key { get; }
    public TValue Value { get; }
}
```

Finding an entry by key needs equality on `TKey`, and the library supplies a default policy for any type through `EqualityComparer<TKey>.Default`, which uses the key type's own `Equals`. The example stores account balances under customer IDs and searches by key.

In [ ]:
using System;
using System.Collections.Generic;

var ledger = new KeyedList<string, decimal>();
ledger.Add("C-104", 1250.75m);
ledger.Add("C-117", 80.00m);
ledger.Add("C-121", 4310.20m);

Console.WriteLine(ledger.TryFind("C-117", out decimal balance) ? $"C-117 owes {balance}" : "C-117 not found");
Console.WriteLine(ledger.TryFind("C-999", out balance) ? $"C-999 owes {balance}" : "C-999 not found");
foreach (Entry<string, decimal> entry in ledger.Entries)
    Console.WriteLine($"{entry.Key}: {entry.Value}");

class Entry<TKey, TValue>
{
    public TKey Key { get; }
    public TValue Value { get; }
    public Entry(TKey key, TValue value) { Key = key; Value = value; }
}

class KeyedList<TKey, TValue>
{
    private readonly List<Entry<TKey, TValue>> entries = new();
    private readonly IEqualityComparer<TKey> keyEquality = EqualityComparer<TKey>.Default;

    public IReadOnlyList<Entry<TKey, TValue>> Entries => entries;

    public void Add(TKey key, TValue value) => entries.Add(new Entry<TKey, TValue>(key, value));

    public bool TryFind(TKey key, out TValue value)
    {
        foreach (Entry<TKey, TValue> entry in entries)
        {
            if (keyEquality.Equals(entry.Key, key))
            {
                value = entry.Value;
                return true;
            }
        }
        value = default!;
        return false;
    }
}

C-117 owes 80.00
C-999 not found
C-104: 1250.75
C-117: 80.00
C-121: 4310.20


The lookup for `C-117` succeeds and reports 80.00; the lookup for `C-999` fails and the method says so through its `bool` result rather than by returning a misleading zero balance. The enumeration then lists all three entries in insertion order. `KeyedList<string, decimal>` is a dictionary ADT implemented as a linear scan: correct, simple, and O(n) per lookup. Chapter 19 replaces the scan with a hash table behind the same `TryFind` contract, which is the representation independence this chapter keeps returning to. The `TryFind` shape, a Boolean result plus an `out` value, is the library's own convention for lookups that may fail, and the one you should copy.

```{index} Ordering and Equality Are Different Contracts
```

## Ordering and Equality Are Different Contracts




```{index} Ordering and Equality Are Different Contracts
```


An ordering comparer answers before, tied, or after with a negative value, zero, or a positive value. An equality comparer answers whether two values identify the same key. A price tie does not make two products the same product. Supply the same ordering policy to sorting and binary search; supply matching Equals and GetHashCode policies to hash collections. Equal keys must have equal hash codes, but unequal keys can collide. Never change a stored key in a way that changes its equality or hash value.


In [1]:
using System.Linq;
using System;
using System.Collections.Generic;
Invoice[] invoices = { new Invoice("A", 30m), new Invoice("B", 10m), new Invoice("C", 30m) };
Array.Sort(invoices, new AmountComparer());
foreach (Invoice invoice in invoices) Console.WriteLine($"{invoice.Id}: {invoice.Amount}");
class Invoice
{
    public string Id { get; }
    public decimal Amount { get; }
    public Invoice(string id, decimal amount) { Id = id; Amount = amount; }
}
class AmountComparer : IComparer<Invoice>
{
    public int Compare(Invoice? left, Invoice? right)
    {
        if (left is null || right is null) throw new ArgumentNullException();
        return left.Amount.CompareTo(right.Amount);
    }
}


B: 10
A: 30
C: 30


The named AmountComparer implements IComparer<Invoice>. Comparing Amount uses decimal.CompareTo, which avoids arithmetic subtraction and conversion. A comparer must be consistent and transitive. Array.Sort does not promise arrival order for tied values. A stable sorting algorithm preserves that order; Chapter 16 makes the distinction visible.


In [1]:
using System.Linq;
using System;
using System.Collections.Generic;
Dictionary<string, int> stock = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
stock.Add("P-100", 4);
Console.WriteLine(stock["p-100"]);
IEqualityComparer<string> identity = StringComparer.OrdinalIgnoreCase;
Console.WriteLine(identity.Equals("P-100", "p-100"));
Console.WriteLine(identity.GetHashCode("P-100") == identity.GetHashCode("p-100"));


4
True
True


```{index} Reading Short Expressions and Queries
```

## Reading Short Expressions and Queries



This bridge is an early exposure to compact syntax used by later examples, not a requirement to learn a second programming model. Read `x => x > 0` as “given x, return whether x is positive.” Recognize the pipeline's operations through the explicit loop you already know; return to the reminder when a later example uses that spelling.

| Step on `3, -1, 2, 0` | Values now described |
| --- | --- |
| `Where(x => x > 0)` | `3, 2` |
| `Select(x => x * 10)` | `30, 20` |
| `OrderBy(x => x)` | `20, 30` |
| `ToArray()` | A separate array containing `20, 30` |

Only `ToArray` stores the pipeline's result here. Defining a query and enumerating it are separate events.


In [1]:
using System;
using System.Collections.Generic;
using System.Linq;
int[] values = { 3, -1, 2, 0 };
var loopResult = new List<int>();
foreach (int value in values)
    if (value > 0) loopResult.Add(value * 10);
loopResult.Sort();
var query = values.Where(x => x > 0).Select(x => x * 10).OrderBy(x => x);
int[] snapshot = query.ToArray();
Console.WriteLine("Loop: " + string.Join(",", loopResult));
Console.WriteLine("Snapshot: " + string.Join(",", snapshot));
values[0] = 4;
Console.WriteLine("Query after edit: " + string.Join(",", query));
Console.WriteLine("Snapshot after edit: " + string.Join(",", snapshot));


Loop: 20,30
Snapshot: 20,30
Query after edit: 20,40
Snapshot after edit: 20,30


The loop and first materialized query agree. Changing `values[0]` changes what a later enumeration of `query` reads, so the third line ends in `20,40`. The already-created snapshot stays `20,30`. A lambda supplies the small operation; `Where`/`Select`/`OrderBy` describe when and how to apply it. The next iterator reading develops suspended and deferred work more fully.

**Prediction:** move the edit before `ToArray`. Both the new snapshot and later query enumeration would then contain `20,40`. Keep compact syntax as a recognition aid; use the explicit loop when learning or debugging the algorithm itself.



```{index} Reading Short Expressions and Queries
```


Later collection examples sometimes use compact C# forms. The expression x => x > 0 is a lambda: a function receiving x and returning the comparison result. A library method can call that function for each value. Where filters values, Select transforms them, OrderBy orders them, and ToArray materializes a separate array. These LINQ extension methods require using System.Linq. Prefer an explicit loop when it makes the algorithm or mutation clearer. Query pipelines can defer work until enumeration; ToArray executes the pipeline now.


In [1]:
using System.Collections.Generic;
using System;
using System.Linq;
int[] arrivals = { 30, -1, 10, 20 };
int[] report = arrivals.Where(value => value >= 0).OrderBy(value => value).Select(value => value * 2).ToArray();
Console.WriteLine(string.Join(", ", report));
Console.WriteLine(string.Join(", ", arrivals));


20, 40, 60
30, -1, 10, 20


An expression-bodied member such as public int Count => items.Count; is shorthand for a getter returning that expression. A target-typed new() uses the declared variable type, so List<int> values = new(); constructs a List<int>. Neither abbreviation changes the data-structure contract. Named classes and explicit constructors remain useful while learning a new model.


```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. `var stack = new ArrayStack<decimal>(); stack.Push(10);` compiles. Why, when 10 is an `int`?

```{dropdown} Answer
The compiler converts the `int` literal to `decimal` implicitly because that conversion never loses information. The reverse, pushing a `decimal` onto an `ArrayStack<int>`, is rejected. Generics do not relax the ordinary conversion rules; they apply them with `T` filled in.
```

2. A generic method `Max<T>` compiles with `where T : IComparable<T>`. A caller passes two `Invoice` objects and gets a compile error. What must change, and where?

```{dropdown} Answer
The `Invoice` class must implement `IComparable<Invoice>`, which means defining `CompareTo`. The fix is in the element type, not in `Max<T>`; the constraint did its job by reporting the missing capability at the call site.
```

3. Why does `Dictionary<TKey, TValue>` need two type parameters while `List<T>` needs one?

```{dropdown} Answer
A list stores one kind of thing and finds it by position, which is always an `int`. A dictionary finds a value by a key that the program chooses, so the key type and the value type are independent decisions and each needs its own parameter.
```

4. What does a generic container give up by being homogeneous, and what is the usual way to get it back?

```{dropdown} Answer
It cannot hold values of unrelated types in one collection. When mixed kinds are genuinely needed, declare an interface or base class that all of them implement and use that as the type argument, so the compiler still knows what every element can do.
```

```{index} Practice
```

## Practice




1. Write a generic `static T First<T>(List<T> list)` that returns the first element or throws `InvalidOperationException` if the list is empty.
2. Write a generic `static void Swap<T>(ref T a, ref T b)` and test it with `int` and `string`.
3. Add a generic `IQueue<T>` interface with `Count`, `Enqueue`, `Dequeue`, and `Peek` members.
4. Explain why `IStack<T>` is a better ADT contract than separate `StringStack`, `IntStack`, and `BookStack` interfaces.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA CS3, Chapter Introduction: Lists](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/ListIntro.html). The worked business scenarios and guided reasoning here are original teaching examples.
